# Phase 3, step 1 — the harness

This notebook walks through the first piece of the **agent harness**: the thing that
actually drives a run, and the reason we build it ourselves instead of plugging into
someone else's coding agent.

You'll drive the whole pipeline **by hand** first. Only at the end does an LLM touch it.
That order is on purpose — everything the agent can do, you will have done yourself, so
there's no magic left in the box.

## 1. Why we own the harness

Up to now the library has had one promise: **leakage-safe by construction**. Not "we
remind you not to leak" — you *structurally cannot*. The sandbox has no model libraries,
so no free-form code can train anything; target encoding only happens through
`fit_encoding`, which fits per fold and stores what it learned.

Here's the problem we hit. The original plan was to ship these as tools for a
general-purpose agent (Claude Code, Codex, ...) to call. But those agents come with
`bash`, a Python interpreter, and a scratchpad. So the easiest path for the model
becomes:

> "twelve lines of sklearn in the scratchpad"

...which goes straight around the sandbox, around the fold discipline, around
`fit_encoding`. The guarantee evaporates — not because anything is broken, but because
there was a *cheaper route*.

**A boundary is only real if our tools are the only way to the data.**

So: we own a thin harness that registers *only* our tools. No shell. No scratchpad. No
free code. The one thing that stays swappable is the **LLM** — Claude, GPT, Gemini, one
config string.

The sandbox doesn't disappear in this picture, by the way — it gets *promoted*. It stops
being one tool among many and becomes the agent's entire scratchpad: expressive enough
to compute features, structurally incapable of training a model or building a leaky one.

### The shape of it

```
   Harness (our code) ── runs the loop ────────── LLM via config
        │                                        (claude / gpt / gemini)
        │ registers ONLY these:
        ├── inspect_data          ← step 1 (this notebook)
        ├── train_tabular_model   ← step 1
        ├── finalize_run          ← step 1
        ├── transform_features    ← next
        ├── fit_encoding          ← next
        ├── blend_models          ← next
        └── predict               ← next
```

We're building the **thin vertical slice** first: inspect → train → finalize. Three
tools, one complete run, end to end. Once that pattern is proven, the other four are
mechanical copies of it.

## 2. Two design rules that shape everything

Before any code, two rules. Nearly every design choice below follows from them.

**Rule 1 — tools speak in file paths, never DataFrames.**

A tool takes a path, writes its results into the run directory, and returns a small
JSON summary plus a path. It never hands the model a table.

Why it matters: the model's context is finite. If a tool returned 1,460 rows, three or
four calls would fill the window and the run would be over. Returning ~15 lines instead
means a run can go for *dozens* of cycles. This is the single property that makes
long, unattended runs possible at all.

**Rule 2 — the run directory is the state.**

Nothing important lives in memory. Models, predictions, metrics, the experiment log —
all files on disk. So a crash, a restart, or wiping the model's context entirely doesn't
lose the run. That last one matters a lot: an overnight run *will* exceed the context
window, and the fix is to throw the conversation away and start a fresh agent pointed at
the same directory.

## 3. Setup — a small dataset

Nothing real; we want something we can reason about completely. `y` genuinely depends on
`x1` and `x2`, `noise` is pure noise, and `city` is a text column — included on purpose,
you'll see why in a moment.

In [ ]:
import os, json, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)

df = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "noise": rng.normal(size=n),
    "city": rng.choice(["berlin", "hamburg", "koeln"], size=n),
    "y": (x1 + 0.5 * x2 + rng.normal(scale=0.3, size=n) > 0).astype(int),
})

workdir = tempfile.mkdtemp(prefix="harness_demo_")
data_path = os.path.join(workdir, "data.csv")
df.to_csv(data_path, index=False)

df.head()

## 4. The session — a handle on one run

`RunSession` is the harness's bookkeeping: where the data is, where to write, and what
we're predicting. It is *not* the state — the state is the directory it points at. This
object could be thrown away and rebuilt from the same three arguments.

`TaskSpec(target="y")` says only *what* we're predicting. Task type (binary?
regression?) and the metric are left blank deliberately — the harness detects them from
the target column and then **pins them for the run**, so every score you compare later
was measured the same way.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="y", n_splits=5),
)

print("run dir:", session.run_dir)
print("resolved:", session.resolved())

## 5. Tool 1 — `inspect_data`

These tools are ordinary Python functions. No LLM involved. That's deliberate: they're
unit-testable, and you can drive the entire pipeline by hand — which is exactly what
we're about to do.

In [ ]:
from tabular_agent.harness.tools import inspect_data, train_tabular_model, finalize_run

profile = inspect_data(session)
print(json.dumps(profile, indent=2))

### Read that output carefully

Three things to notice.

**It's small.** That entire response is a few hundred tokens. That's Rule 1 doing its
job — and it's why a run can keep going for dozens of steps.

**It decided the task.** `task_type` and `metric` were detected from `y` and are now
locked in for the run. Every score from here on is comparable.

**`non_numeric_columns` lists `city`.** The tree models take a numeric matrix, so a raw
text column would crash the fit. We could have quietly cast it to numbers — but a
careless cast is precisely how train/serve skew gets in. So the tool **drops it and says
so out loud**. The proper way to use `city` is `fit_encoding`, which learns its mapping
per fold and stores it. That's the next slice; for now the honest thing is to say
"there's signal here you aren't using."

## 6. The journal — memory that survives a context wipe

`inspect_data` wrote a file as a side effect. This is `run_journal.json`, and it's how a
run survives losing its conversation.

In [ ]:
from tabular_agent.harness import journal

print(json.dumps(journal.load(session.run_dir), indent=2))

It's mostly empty right now — only the task is set. The important part is *who writes to
it*.

The naive design is "the agent keeps notes." That fails, and it fails in a boring way:
the agent forgets. Somewhere around cycle 30, exhausted and full of context, it just
stops logging. Then you compact and the run's history is gone.

So we split by **who actually has the information**:

| Layer | Written by | When |
|---|---|---|
| `leaderboard` — the facts | the training tool | automatically, whenever a model is actually trained |
| `log` — the interpretation | the agent | one short note per cycle: what I tried, why, what next |

The leaderboard is a **by-product of the experiment**. There is no code path that trains
a model without recording it, so it takes zero discipline and can't be skipped. Only the
interpretation layer depends on the agent, and if the agent forgets a note, the facts
are still all there.

## 7. Tool 2 — `train_tabular_model`

First the baseline. This is the model that ignores every feature and just predicts the
overall rate. It sounds pointless. It is the most important number in the run.

Without it, "CV score 0.71" is unreadable. If the baseline is 0.69, you've learned
almost nothing. Any model that can't beat the baseline isn't a weak model — it's a bug,
leakage, or an unlearnable target.

In [ ]:
baseline = train_tabular_model(session, model_type="baseline")
print(json.dumps(baseline, indent=2))

In [ ]:
lgbm = train_tabular_model(session, model_type="lgbm")
print(json.dumps(lgbm, indent=2))

### How to read a result

- **`cv_score`** — the mean across the 5 folds.
- **`higher_is_better`** — tells you which direction is good. `logloss` is lower-better;
  `auc` is higher-better. Without this the agent would confidently pick the worst model.
- **`fold_spread`** — the standard deviation across folds. This is the one people skip.
  A great mean with a huge spread is an *unstable* model, not a good one — it means the
  score depended heavily on which rows happened to land in which fold.
- **`vs_baseline`** — the number that says whether any of this was worth it.

In [ ]:
print(f"baseline : {baseline['cv_score']:.4f}")
print(f"lgbm     : {lgbm['cv_score']:.4f}   (spread {lgbm['fold_spread']:.4f})")
print(f"metric   : {lgbm['metric']}  |  higher is better: {lgbm['higher_is_better']}")

### An honest look at what just happened

When this was written, the run came out roughly like this (yours will be close):

```
baseline : 0.6907   spread 0.0007
lgbm     : 0.6053   spread 0.2115     <- per fold: 0.25, 0.67, 0.72, 0.51, 0.87
```

Lower is better for `logloss`, so lgbm *does* beat the baseline. Good.

But look at the folds: **0.25 on one, 0.87 on another.** That's more than a threefold
difference depending on which rows happened to be held out. The mean is real, but the
model is nowhere near as solid as "0.605 beats 0.691" makes it sound — with 400 rows and
200 trees, each fold is fitting quite a different model.

This is the exact failure the `fold_spread` field exists to expose, and it's worth
sitting with, because it's the whole reason a single CV number is dangerous on its own.
An agent optimising on the mean alone would happily declare victory here. The right read
is: "beats baseline, but unstable — get more data, or fewer trees, before trusting it."

That's also why the system prompt tells the agent to report spread, not just score.

## 8. What's actually on disk

Rule 2 in the flesh. Everything below is reconstructable without the notebook, the
process, or the conversation.

In [ ]:
for root, dirs, files in os.walk(session.run_dir):
    depth = root[len(session.run_dir):].count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        print("  " * (depth + 1) + f)

In [ ]:
j = journal.load(session.run_dir)
print("baseline ->", j["baseline"])
print("best     ->", j["best"])
print()
for row in j["leaderboard"]:
    print(f"{row['id']}  {row['model_type']:<9} {row['cv_score']:.4f}  spread={row['fold_spread']:.4f}")

Notice `best` points at `e2` (the lgbm), not `e1`. The baseline is tracked in its own
field precisely so it can serve as the yardstick without ever being mistaken for the
deliverable.

## 9. Tool 3 — `finalize_run`

Called **once**, at the end. It reads the journal's `best` pointer, packages that
experiment into a deployable bundle, and writes the human-facing report.

Why once, and why separate from the journal? Because they answer different questions.
The journal is scratch memory for the *next agent* — updated constantly, machine-shaped.
`finalize_run` is the deliverable for the *human* — written once, final. Merging them
would mean re-packaging a deployment bundle on every cycle: expensive, and it muddles
"my notes" with "the thing I ship."

In [ ]:
final = finalize_run(session)
print(json.dumps(final, indent=2))

In [ ]:
print(open(os.path.join(session.run_dir, final["report_path"])).read())

## 10. Does the bundle actually work?

A report is a claim. The bundle is the product. Let's load it fresh from disk — as a
deployment would — and predict on raw rows with **no target column**, which is the real
test: anything that secretly needed the target would fail here.

In [ ]:
from tabular_agent.io import load_table
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, final["bundle_dir"]))
raw = load_table(data_path).drop(columns=["y"])   # no target, like production

preds = predict_bundle(bundle, raw)
print("predicted", len(preds), "rows; first five:", np.round(preds[:5], 4))

That's a complete run: inspect → baseline → model → finalize → deployable predictor.
**Every step of it was a plain function call.** No LLM has been involved so far.

## 11. Now the agent

Here's the payoff. The harness does exactly one thing: it hands those same three
functions to a model, and nothing else.

In [ ]:
from tabular_agent.harness.agent import build_agent, DEFAULT_MODEL

agent = build_agent()
print("model:", DEFAULT_MODEL)
print("tools the agent can call:", sorted(agent._function_toolset.tools))

**That list is the entire guarantee.**

There is no `bash`. No `read_file`. No `run_python`. No way to get the raw table into
the model's context. So "don't train a model outside `train_tabular_model`" isn't a rule
we ask it to follow — it's the only thing it *can* do.

Compare the two designs:

| | Prompt-enforced | Tool-enforced (ours) |
|---|---|---|
| "Only train via our tool" | a request the model may ignore | there is no other tool |
| "Don't build leaky features" | a request | the sandbox has no target column |
| Under pressure / long runs | degrades | unchanged |

This is also why there's a test asserting the tool list is *exactly* these three. If
someone adds a convenience `read_file` tool six months from now, the test fails and
makes them say out loud that they're reopening the door.

### The system prompt gets a much smaller job

Because the boundary is structural, the prompt no longer has to police anything. It only
supplies the *order* of work and how to read results.

In [ ]:
from tabular_agent.harness.instructions import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

## 12. Optional — a live run

This is the only cell that calls a paid API. It needs `ANTHROPIC_API_KEY` set. Skip it
freely; nothing below depends on it.

Watch what the agent does: it should inspect, notice `city` is unusable, train a
baseline, train a real model, compare against the baseline, and finalize.

In [ ]:
# Requires ANTHROPIC_API_KEY. Uncomment to run.
#
# from tabular_agent.harness.agent import run
#
# result = run(session, "Build the best model you can for this dataset, then finalize.")
# print(result.output)

## 13. Where this goes next

Done in this step:

- the file-based tool contract (paths in, small summaries out)
- the run directory as the state
- `run_journal.json` — auto-written facts + a slot for agent interpretation
- three tools, one full run, LLM-optional
- the constrained tool surface, asserted by a test

Next:

1. **The other four tools** — `transform_features`, `fit_encoding`, `blend_models`,
   `predict`. Same adapter pattern; this is where `city` finally becomes usable.
2. **The agent's note layer** — `journal_append`, so the *why* survives alongside the
   *what*.
3. **Compaction and stopping** — the harness watches the token budget, wipes the
   conversation, and restarts a fresh agent against the same `run_journal.json`. It also
   owns the stop decision ("no progress in K cycles, or out of time") so the run can't
   loop forever. This is what makes overnight runs work.
4. **Multi-provider config** — GPT and Gemini alongside Claude.